# TopK search and memory tools for LlamaIndex agents

[TopK](https://topk.io) is a search engine for AI agents. It runs keyword (BM25), semantic, vector and hybrid search, plus metadata filters, in one query against one index, and it can embed text on the server.

This notebook gives a LlamaIndex `FunctionAgent`:

1. **Search tools** (`TopKToolSpec`): `keyword_search`, `semantic_search` and `hybrid_search`, with descriptions that tell the agent when to use each and how to filter.
2. **Long-term memory tools** (`TopKMemoryToolSpec`): `remember`, `recall` and `forget`, stored in an isolated TopK partition per user.
3. **A memory block** (`TopKMemoryBlock`) for LlamaIndex `Memory`, so flushed chat history becomes searchable long-term memory.

It also shows `TopKVectorStore` as a regular `VectorStoreIndex` backend.

## Setup

In [ ]:
%pip install llama-index-tools-topk llama-index-vector-stores-topk llama-index-llms-anthropic

Create an API key at [console.topk.io](https://console.topk.io) and pick a [region](https://docs.topk.io/regions). TopK embeds text on the server, so you don't need an embedding model. The agent below uses Claude; any function-calling LLM works.

In [ ]:
import os

os.environ["TOPK_API_KEY"] = "..."
os.environ["TOPK_REGION"] = "aws-us-east-1-elastica"
os.environ["ANTHROPIC_API_KEY"] = "..."

In [ ]:
from llama_index.llms.anthropic import Anthropic

llm = Anthropic(model="claude-sonnet-5-5")

## Load a runbook into TopK

Four of these error codes differ by one transposed digit. Semantic search can't reliably tell `E-4021` from `E-4012`; BM25 can.

In [ ]:
from llama_index.core.schema import TextNode
from llama_index.vector_stores.topk import TopKVectorStore

store = TopKVectorStore(collection_name="runbook")  # server-side embeddings

runbook = [
    ("E-4021", "Error code E-4021 means the payment gateway timed out. Retry after 30 seconds.", "payments", 2024),
    ("E-4012", "Error code E-4012 means the card was declined by the issuing bank.", "payments", 2024),
    ("E-4201", "Error code E-4201 means the shipping address failed validation.", "checkout", 2023),
    ("E-2041", "Error code E-2041 means the inventory service returned stale stock levels.", "inventory", 2025),
    (
        "slow-pages",
        "Pages render slowly when the CDN cache is cold after a deploy; warm it with the prefetch job.",
        "performance",
        2025,
    ),
    ("churn", "Customers cancel subscriptions mostly because onboarding takes too long.", "product", 2024),
]
store.add([TextNode(id_=i, text=t, metadata={"topic": topic, "year": y}) for i, t, topic, y in runbook])

## Search tools

`corpus_description` and `metadata_fields` are added to every tool description, so the agent knows what it's searching and what it can filter on.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent, ToolCall

from llama_index.tools.topk import TopKToolSpec

search_tools = TopKToolSpec(
    store,
    corpus_description="an engineering runbook: error codes, incidents and product notes",
    metadata_fields={"topic": "payments, checkout, inventory, performance or product", "year": "integer"},
).to_tool_list()

for tool in search_tools:
    print(tool.metadata.name, "->", tool.metadata.description.split("\n")[2].strip())

In [ ]:
agent = FunctionAgent(
    tools=search_tools,
    llm=llm,
    system_prompt="You answer questions using the search tools. Always search before answering. Be brief.",
)


async def ask(agent, question):
    handler = agent.run(question)
    async for event in handler.stream_events():
        if isinstance(event, ToolCall):
            print(f"-> {event.tool_name}({event.tool_kwargs})")
    print(await handler)

An exact error code: the agent picks `keyword_search`.

In [ ]:
await ask(agent, "What does error E-4021 mean?")

A paraphrased question that shares no keywords with the answer: the agent picks semantic or hybrid search.

In [ ]:
await ask(agent, "Why does the site feel sluggish right after we ship a release?")

A question with constraints: the agent adds a metadata filter.

In [ ]:
await ask(agent, "List the payments error codes from 2024 only.")

## Long-term memory per user

`TopKMemoryToolSpec` stores memories in a TopK partition named after the user. Partitions are fully isolated, so one user's agent can't read another user's memories, and there's no `user_id` filter to forget. `recall` uses hybrid search, so it matches paraphrases ("what can't I eat?") and exact terms (an order number).

In [ ]:
from llama_index.tools.topk import TopKMemoryToolSpec

memories = TopKVectorStore(collection_name="agent-memory")


def assistant_for(user_id):
    return FunctionAgent(
        tools=TopKMemoryToolSpec(memories, user_id=user_id).to_tool_list(),
        llm=llm,
        system_prompt="You are a personal assistant with long-term memory tools. Be brief.",
    )

In [ ]:
await ask(assistant_for("user-alice"), "Please remember that I'm allergic to peanuts.")

A new agent run with no shared chat history recalls it:

In [ ]:
await ask(assistant_for("user-alice"), "Suggest a snack for me. Check what you know about me first.")

Bob's agent searches Bob's partition only:

In [ ]:
await ask(assistant_for("user-bob"), "Do I have any food allergies? Check your memory.")

## A TopK memory block for `Memory`

`TopKMemoryBlock` plugs into LlamaIndex's `Memory`. When short-term chat history overflows, old messages are flushed to the user's TopK partition. On each turn the block runs a hybrid search over them with the latest messages and adds the hits to the prompt. Memories carry across sessions because the partition is per user, not per session.

In [ ]:
from llama_index.core.memory import Memory

from llama_index.tools.topk import TopKMemoryBlock

memory = Memory.from_defaults(
    session_id="chat-1",
    token_limit=4000,
    memory_blocks=[TopKMemoryBlock(vector_store=memories, user_id="user-alice", similarity_top_k=3)],
)
chat_agent = FunctionAgent(tools=[], llm=llm)
response = await chat_agent.run("I live in Lisbon and I'm vegetarian.", memory=memory)

## TopK as a `VectorStoreIndex` backend

The same store works as a regular LlamaIndex vector store, with `vector_store_query_mode` set to `"default"`, `"text_search"` (BM25) or `"hybrid"`. `TopKServerSideEmbedding` stops LlamaIndex from calling an embedding model, because TopK embeds on the server.

In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.topk import TopKServerSideEmbedding

index = VectorStoreIndex.from_vector_store(store, embed_model=TopKServerSideEmbedding())

for mode in ["text_search", "default", "hybrid"]:
    nodes = index.as_retriever(vector_store_query_mode=mode, similarity_top_k=2).retrieve("checkout payment E-4021")
    print(mode, [n.node.node_id for n in nodes])

## Clean up

In [ ]:
store.client.collections().delete("runbook")
store.client.collections().delete("agent-memory")

## Learn more

- [`llama-index-tools-topk`](https://github.com/topk-io/llama-index-tools-topk) and [`llama-index-vector-stores-topk`](https://github.com/topk-io/llama-index-vector-stores-topk) on GitHub
- [TopK docs](https://docs.topk.io), including [true hybrid search](https://docs.topk.io/guides/true-hybrid-search) and [partitions](https://docs.topk.io/concepts#multi-tenancy)